# 随机梯度下降 SGD (Stochastic Gradient Descent)

* 之前的梯度下降，我们把所有的 $x_i$, 所有的 $\beta_i$ 都纳入了计算，当数据量十分庞大时，就不太可能这么干了，
* 于是，我们需要在众多变量中随机挑选一些出来，于是有了随机梯度下降

In [104]:
import numpy as np
from sklearn import pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [105]:
# 1. 定义数据
# 自变量，
n = 10000
X = np.arange(1, n + 1).reshape(-1, 1)
# 因变量，数学考试成绩
real_coef = 2.0
real_intercept = 1.0
np.random.seed(42)
y = X.flatten() * real_coef + real_intercept + np.random.randn(n) * 0.05
X, y

(array([[    1],
        [    2],
        [    3],
        ...,
        [ 9998],
        [ 9999],
        [10000]], shape=(10000, 1)),
 array([3.02483571e+00, 4.99308678e+00, 7.03238443e+00, ...,
        1.99969647e+04, 1.99990248e+04, 2.00010322e+04], shape=(10000,)))

In [108]:

sgd_model = SGDRegressor(
    loss="squared_error",        # 损失函数，默认为均方误差
    fit_intercept=True,         # 是否计算截距
    learning_rate="constant",   #  学习率是否恒定（如果收敛了，是否可以慢慢变小）
    eta0=0.1,           # 初始学习率
    max_iter=10**9,     # 最大迭代次数
    tol=1e-8,           # 损失值变化量小宇 tol 时停止迭代
    penalty="l1",       # 正则化为 L1 正则化，防止复杂模型过拟合，惩罚参数个数过多的方案
    alpha=0.0001,       # 正则化强度
)
sgd_model.fit(X, y)
print(sgd_model.coef_)
print(sgd_model.intercept_)

[-5.84549803e+13]
[-9.60386037e+12]


非常巨大且离谱的数据，不是简单震荡能解释的，让我们看看发生了什么

In [95]:
sgd_model = SGDRegressor(
    loss="squared_error",        # 损失函数，默认为均方误差
    fit_intercept=True,         # 是否计算截距
    learning_rate="constant",   #  学习率是否恒定（如果收敛了，是否可以慢慢变小）
    eta0=0.1,           # 初始学习率
    max_iter=1,     # 最大迭代次数,
    penalty="l1",       # 正则化为 L1 正则化，防止复杂模型过拟合，惩罚参数个数过多的方案
    alpha=0.0001,       # 正则化强度
)
sgd_model.fit(X, y)
print(f"第一次迭代后: coef={sgd_model.coef_}, intercept={sgd_model.intercept_}")

第一次迭代后: coef=[-1.67978703e+14], intercept=[-1.64001952e+13]


/opt/anaconda3/envs/machine-learn-fundamental/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


第一次迭代就走远了，之后学习步长太远，走不回来了

那为什么第一次走远了？

In [53]:
def g(X, y, beta, n):
    return X.T @ (X @ beta - y) * 2 / n

feature_n = X.shape[1]
beta0 = np.zeros(feature_n)
g(X, y, beta0, feature_n)

array([-1.33363331e+12])

第一步偏导就非常的大

根据

$$\nabla J(\beta) = \frac{2}{n} X^T (X\beta - y)$$

当 $\beta_0 = 0$ 时，

$$\nabla J(\beta) = -\frac{2}{n} X^T y$$

$X$ 和 $y$ 都是 1～10000 的数字，所以

$$
\nabla L(0) \approx -\frac{2}{10000} \left( 2 \cdot \frac{10000 \cdot 10001 \cdot 20001}{6} + \frac{10000 \cdot 10001}{2} \right)
\approx -1.33 \times 10^8
$$

解决这个问题，需要数据标准化

In [96]:
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

sgd_model = SGDRegressor(
    loss="squared_error",  # 损失函数，默认为均方误差
    fit_intercept=True,  # 是否计算截距
    learning_rate="constant",  #  学习率是否恒定（如果收敛了，是否可以慢慢变小）
    eta0=0.1,  # 初始学习率
    max_iter=10 ** 6,  # 最大迭代次数
    tol=1e-5,  # 损失值变化量小宇 tol 时停止迭代
    penalty="l1",  # 正则化为 L1 正则化，防止复杂模型过拟合，惩罚参数个数过多的方案
    alpha=0.0001,  # 正则化强度
)
# 1. 定义数据
# 自变量，
n = 10000
X = np.arange(1, n + 1).reshape(-1, 1)
# 因变量，数学考试成绩
real_coef = 2.0
real_intercept = 1.0
np.random.seed(42)
y = X.flatten() * real_coef + real_intercept + np.random.randn(n) * 0.05

# 关键
scaler = StandardScaler()
X = scaler.fit_transform(X)

sgd_model.fit(X, y)
print(sgd_model.coef_[0])
print(sgd_model.intercept_[0])

5773.512592591134
10002.006910857861


小了不少，但是依旧很大，y 没有被标准化，所以 $-\frac{2}{n} X^T y$ 依旧很大

In [98]:
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

sgd_model = SGDRegressor(
    loss="squared_error",  # 损失函数，默认为均方误差
    fit_intercept=True,  # 是否计算截距
    learning_rate="constant",  #  学习率是否恒定（如果收敛了，是否可以慢慢变小）
    eta0=0.1,  # 初始学习率
    max_iter=10 ** 6,  # 最大迭代次数
    tol=1e-5,  # 损失值变化量小宇 tol 时停止迭代
    penalty="l1",  # 正则化为 L1 正则化，防止复杂模型过拟合，惩罚参数个数过多的方案
    alpha=0.0001,  # 正则化强度
)
# 1. 定义数据
# 自变量，
n = 10000
X = np.arange(1, n + 1).reshape(-1, 1)
# 因变量，数学考试成绩
real_coef = 2.0
real_intercept = 1.0
np.random.seed(42)
y = X.flatten() * real_coef + real_intercept + np.random.randn(n) * 0.05

# 关键
scaler_x = StandardScaler()
X = scaler_x.fit_transform(X)
scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

sgd_model.fit(X, y)
print(sgd_model.coef_)
print(sgd_model.intercept_)

[0.99990937]
[1.17787593e-05]


收敛到了 coef = 1, intercept = 0, 因为 x y 都进行了标准化，所以 y = 2x 的比例关系就没有了

想看原始尺寸的系数，一定要反标准化

In [101]:
coef_pred = sgd_model.coef_ * scaler_y.scale_[0] / scaler_x.scale_[0]
print(coef_pred)

[1.99981857]


完整反推原始系数

In [119]:
from sklearn.pipeline import make_pipeline
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

sgd_model = SGDRegressor(
    loss="squared_error",  # 损失函数，默认为均方误差
    fit_intercept=True,  # 是否计算截距
    learning_rate="constant",  #  学习率是否恒定（如果收敛了，是否可以慢慢变小）
    eta0=0.1,  # 初始学习率
    max_iter=10 ** 6,  # 最大迭代次数
    tol=1e-5,  # 损失值变化量小宇 tol 时停止迭代
    penalty="l1",  # 正则化为 L1 正则化，防止复杂模型过拟合，惩罚参数个数过多的方案
    alpha=0.0001,  # 正则化强度
)

n = 10000
X = np.arange(1, n + 1).reshape(-1, 1)
real_coef = 2.0
real_intercept = 1.0
np.random.seed(42)
y = X.flatten() * real_coef + real_intercept + np.random.randn(n) * 0.05

pipeline = make_pipeline(StandardScaler(), sgd_model)
pipeline.fit(X, y)

# 还原到原始尺度
scaler = pipeline[0]
w_scaled = pipeline[1].coef_[0]
b_scaled = pipeline[1].intercept_[0]

w_original = w_scaled / scaler.scale_[0]
b_original = b_scaled - w_original * scaler.mean_[0]

print(w_original)
print(b_original)

2.0000034397013287
0.9897106313674158


一般做预测的时候不需要知道真实的斜率和截距，所以没那么麻烦，直接 transform 后 predict

| 算法名称 | 每次使用的数据量 | 计算速度 | 梯度准确性 | 收敛稳定性 | 适用场景 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **BGD** (批量梯度下降) | 全部训练样本 | 最慢 | 最准确（真实梯度） | 最稳定，必达局部最优 | 小数据集、凸优化问题 |
| **SGD** (随机梯度下降) | 1个随机样本 | 最快 | 最不准确（噪声大） | 不稳定，容易震荡 | 大数据集、在线学习 |
| **Mini-batch GD** (小批量梯度下降) | 一小批样本（如32/64/128个） | 较快 | 较准确（适中） | 较稳定，介于两者之间 | 绝大多数深度学习任务（最常用） |